# 05 · Intermediate · Your own checks in a few lines

A check is a plain function that returns `True` (pass), `False` (fail), or `None`
(data unavailable). `define_scanner` turns a list of checks into a scanner you can run
next to the built-in ones.

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.3.0-py3-none-any.whl

In [ ]:
from lakehouse_gc import check, define_scanner, fail, ok, rules_dataframe, run, unknown

POLICY_PATH = "/Volumes/main/default/lakehouse_gc/policy.yaml"


@check(
    rule="has_cost_center",  # default: function name -> rule id "tags.has_cost_center"
    title="Has a cost center tag",  # default: built from the function name
    description="Every table needs a cost_center tag for chargeback.",  # default: docstring
    severity="low",  # info | low | medium | high (default: medium)
    dimension="governance",  # governance | cost | ["governance", "cost"]
    on_fail="tag_owner",  # review | tag_owner | archive_candidate | pause_candidate |
    #                       delete_candidate (default: review)
)
def has_cost_center(table, config):  # config is optional: def has_cost_center(table)
    if table["tags"] is None:
        return unknown("Tags could not be read.")  # or just: return None
    if config["required_tag"] in table["tags"]:
        return ok("Cost center present.")  # or just: return True
    return fail(f"Missing tag {config['required_tag']}.")  # or just: return False


@check  # bare decorator: all defaults
def has_owner(table):
    """Tables need an owner."""
    return bool(table["owner"])


@check(severity="medium", dimension="cost", on_fail="archive_candidate")
def recently_written(table):
    if table["days_since_write"] is None:
        return None
    # inferred=True: indirect evidence -> lower confidence -> candidates become review
    return ok(inferred=True) if table["days_since_write"] <= 365 else fail(inferred=True)


def load_tables(config):  # resources may be a list, or a function (optionally taking config)
    # In a real notebook, for example:
    #   return [r.asDict() for r in spark.sql("SELECT ... FROM ...").collect()]
    return [
        {
            "name": "main.sales.orders",  # required: full name (exclusions match on it)
            "id": "main.sales.orders",  # optional, default: name
            "type": "table",  # optional, default: resource_type below
            "scope": "my-workspace",  # optional, default: "default"
            "owner": "team-sales@example.com",  # optional
            "tags": {"cost_center": "cc1"},  # any other keys are passed to your checks
            "days_since_write": 3,
        },
        {"name": "main.tmp.scratch", "owner": None, "tags": {}, "days_since_write": 500},
        {"name": "main.legacy.unknown", "owner": None, "tags": None, "days_since_write": None},
    ]


tags = define_scanner(
    "tags",  # lowercase letters, digits, underscores
    resources=load_tables,
    checks=[has_cost_center, has_owner, recently_written],
    config={"required_tag": "cost_center"},  # defaults; override in policy under rules.tags
    resource_type="table",
)

In [ ]:
display(rules_dataframe(spark, POLICY_PATH, scanners=[tags]))

result = run(POLICY_PATH, scanners=["uc", tags], spark=spark)
print(result.summary())
display(result.to_dataframe(spark))